In [1]:
using LowLevelFEM, LinearAlgebra, IterativeSolvers, Preconditioners, IncompleteLU

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 6 for different dependency version already loaded, 1 for different Julia build configuration, 2 for file size changed)
Precompiling packages...
 378.0 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 383 seconds. 107 already precompiled.
[ Info: Precompiling LowLevelFEMIterativeSolversExt [55978d5f-b2ed-541f-a526-69ba3ceeb2d1] (caches not reused: 6 for different dependency version already loaded, 3 for different Julia build configuration)
Precompiling packages...
   3.5 s  ✓ LowLevelFEM → LowLevelFEMIterativeSolversExt
  1 dependency successfully precompiled in 7 seconds. 110 already precompiled.
[ Info: Precompiling LowLevelFEMPreconditionersExt [731aa7ef-afda-5ca4-bfd0-d21a0237399b] (caches not reused: 7 for different dependency version already loaded, 2 for different Julia build configuration)
Precompiling packages...
   4.7 s  ✓ LowLevelFEM → LowLevelFEMPreconditionersExt
  1 depe

In [2]:
openGeometry("tarto.geo")

## Anyagállandók megadása, feladat kitűzése

In [3]:
mat = Material("part", E=2e5, ν=0.3)
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

## Az erő támadáspontja a furat súlypontjában

In [4]:
A = ∫(U, "hole2", (x,y,z)->1)
Sx = ∫(U, "hole2", (x,y,z)->x)
Sy = ∫(U, "hole2", (x,y,z)->y)
Sz = ∫(U, "hole2", (x,y,z)->z)

xs, ys, zs = Sx/A, Sy/A, Sz/A

(4.005804115633015e-6, -8.595785970749574e-7, 130.0000269311167)

In [5]:
Premote = gmsh.model.occ.addPoint(xs, ys, zs)
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(0, [Premote], -1, "remote")

gmsh.model.mesh.clear()
gmsh.model.mesh.generate(3)

In [6]:
mat_r = Material("remote")
U = Field([mat, mat_r], type=:VectorField, dim=3, fieldName=:u);
Φ = Field([mat, mat_r], type=:VectorField, dim=3, fieldName=:φ);

In [7]:
Ku = ∫(SymGrad(U)' ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="part");

In [8]:
fu = ∫(U ⋅ [0, -1000, 0], Γ="remote")
fφ = ∫(Φ ⋅ [0, 0, 0], Γ="remote");

In [9]:
supp1 = BoundaryCondition("base", uz=0, field=U)
supp2 = BoundaryCondition("hole1", ux=0, uy=0, field=U)
supp3 = BoundaryCondition("perim", uz=-0.00001, field=U);

In [10]:
mpc_u = MultiPointConstraint(master="remote", slave="hole2", field=U)
mpc_φ = MultiPointConstraint(master="remote", slave="hole2", field=Φ)
R = rigidRotationMap(mpc_u, mpc_φ);

In [11]:
K = SystemMatrix([Ku Ku * R; R' * Ku R' * Ku * R]);
F = SystemVector([fu, fφ]);

In [12]:
u0, φ = solveField(Symmetric(K), F, support=[supp1, supp2, supp3], mpc=[mpc_u, mpc_φ]);

In [13]:
u = u0 + R * φ;

In [14]:
showDoFResults(fu, name="f")
showDoFResults(fφ, name="m")
showDoFResults(u, name="u", visible=true, factor=1);

In [15]:
u = nodesToElements(u, onPhysicalGroup="part");

In [16]:
ε = (u ∘ ∇ + ∇ ∘ u) / 2
showStrainResults(ε, name="ε");

In [17]:
E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν/(1-2ν) * trace(ε) * I)
showDoFResults(σ, name="σ");

In [18]:
λ, Q = eigen(σ);

In [19]:
λ1 = λ[1]
λ2 = λ[2]
λ3 = λ[3];

In [20]:
σ_Mohr = λ1 - λ3
showElementResults(σ_Mohr, name="Mohr");

In [21]:
σ_HMH = √(((λ1 - λ2)^2 + (λ2 - λ3)^2 + (λ3 - λ1)^2) / 2)
showElementResults(σ_HMH, name="von Mises");

In [22]:
showElementResults(σ, :z, name="σz")
showElementResults(σ_Mohr - σ_HMH, name="diff")
showDoFResults((σ_Mohr - σ_HMH) / σ_Mohr * 100, name="rel. diff - smooth")

9

In [23]:
b = -σ⋅∇
showElementResults(b, name="b")

10

In [24]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
